# 03 — Tiny GPT v3: load & chat (no retraining)

Notebook **02** *builds and trains* the from-scratch GPT (~30 min). This notebook is the **consumer**: it **loads the saved checkpoint in ~1 second** and lets you generate / chat as much as you want — no training, ever.

**Prerequisite — mint the checkpoint once.** If you've never saved one, run this in a terminal (it's the headless twin of 02's training cell):

```bash
uv run python notebooks/train_v2_checkpoint.py        # ~30 min, one time
```

That writes `notebooks/checkpoints/tiny_gpt_v2/` (weights + tokenizer + config). After that, just re-run this notebook whenever you want to play.

> **Reality check:** this is a *TinyStories* model, not an instruction-tuned assistant. It doesn't answer questions — it *continues* text. Feed it a story opener ("Once upon a time…", "The dragon looked at the boy and said…") rather than "What is 2+2?".

In [1]:
import time
import tiny_gpt   # local module (notebooks/tiny_gpt.py) — the model class + load/generate

t0 = time.time()
model, tok, cfg = tiny_gpt.load("checkpoints/tiny_gpt_v2")
print(f"loaded in {time.time()-t0:.2f}s  |  config = {vars(cfg)}")

loaded in 0.01s  |  config = {'block_size': 256, 'n_embd': 384, 'n_head': 6, 'n_layer': 6, 'vocab_size': 8192, 'eos_token': '<|endstory|>'}


## Generation playground

`tiny_gpt.generate(model, tok, cfg, prompt, n_new=..., temperature=...)` returns a full completion. Re-run this cell with different prompts as often as you like — it reuses the model already in memory.

In [2]:
for prompt in ["Once upon a time", "The dragon looked at the boy and said", "In the dark forest"]:
    print("="*70)
    print(tiny_gpt.generate(model, tok, cfg, prompt, n_new=150, temperature=0.8))
    print()

Once upon a time, there was a little girl named Lily. She loved to play with her toys and eat cookies. One day, her mom gave her a big box full of cookies to eat. Lily was so happy and ate all the cookies with her.

When she was finished cooking, her mom asked her to eat her a cookie. Lily didn't want her cookie or her mom said she had to finish some. Her mom added a cookie and they sat on a plate. Lily loved her mom and thought her favorite salad was a yummy snack.

As they were eating their cookies, Lily's mom told her that they were going to have some for lunch before lunch. Lily was excited and ran outside to eat the cookies. She loved seeing all the

The dragon looked at the boy and said, "I'm not sad. I can only see my dragon."

The dragon smiled and said, "I promise, he's very curious. I'll teach you many words on the castle." The dragon leapt into her castle and the princess in a few seconds before she could touch the dragon.

Finally, the dragon said, "Do you?"

The dragon sai

In the dark forest, there were many animals, but they were also very quiet. They had big, shiny, black and white feathers. Tom and Lily were also amazed.

"Look, Tom, this is the best thing I see," Lily said, pointing to a long stick and swues.

"Wow, it is so cool! We have a turn!" Tom said, grabbing Lily's hand.

“That is very big and heavy, Lily. But I made a loud sound and scary animal. It is the best animal in the world. I can teach you how to use things and learn new things. You can also spell your powers and learn. You can also learn from them and be kind to each other. You can learn



## Temperature sweep — the one knob worth feeling

Same prompt, rising temperature. Low (~0.4) is coherent but repetitive; high (~1.2) is creative but loopier. (Recall from the handoff: temperature only *bites* on a confident model — v2 is trained enough to show the spread.)

In [3]:
for temp in (0.4, 0.7, 1.0, 1.2):
    print(f"--- temperature {temp} ---")
    print(tiny_gpt.generate(model, tok, cfg, "Once upon a time", n_new=120, temperature=temp))
    print()

--- temperature 0.4 ---
Once upon a time, there was a little boy named Timmy. Timmy loved to play outside and explore the world around him. One day, Timmy went to the park to play with his friends. He saw a big tree and wanted to climb it. 

Timmy said, "I can't climb the tree, but I can't reach it." 

His friends said, "Don't worry, Timmy. Just be careful and be careful." 

Timmy felt happy and continued to climb the tree. He climbed up the tree and saw the tree was tall and tall. He climbed the tree and saw the trees

--- temperature 0.7 ---


Once upon a time, there was a little girl named Lily. She loved to play outside and play. One day, she found a big nation with a big factory. She was scared, but she didn't know what was in the factory.

Suddenly, a kind man came to Lily and said, "Don't worry, little girl. We can fix it together." Lily was happy to hear that and started hammering a lot.

The man was so happy and said, "Thank you, Lily! You're very kind." Lily smiled and said, "I love playing with you too, and

--- temperature 1.0 ---
Once upon a time, there was a little girl named Lily who loved to dress up. She liked to wear her purple dresses and put them on her pretty dress. She made funny faces and giggled when she was bored too!

One day, Lily's mom took her to a party, but Lily didn't want to choose more pretty toys. She said, "Mommy, can we play now?" Her mom said, "No, they are not available for her."

After the party, Lily went back to her room and let her start to dress up. She put on her pink dress and shoe

Once upon a time, there was a little girl named Lily. She had a teddy bear named Teddy. One day, Teddy went to play with his friends. Lily wanted to play with Teddy's whip, but didn't want it to shut. She looked everywhere for him, but she didn't see it outside in the bushes. 

Lily decided to take her bag and Lily showed him to him how she forgave him. He was so gratefulLily and passing the glove to Mittens. They all had fun together and became good friends. From that day on, they played outside and catch paid wear their toys to each other



## Interactive chat

Run the cell below and type prompts at the box that appears. Tokens stream in as they're generated. Type `/quit` to stop, `/temp 0.6` or `/tokens 150` to adjust on the fly.

*(Prefer a real terminal? `uv run python notebooks/chat.py` gives the same REPL outside Jupyter.)*

In [ ]:
temp, ntok = 0.8, 200
print("Type a prompt (/temp N, /tokens N, /quit to stop).")
while True:
    prompt = input("\nyou \u25b8 ").strip()
    if not prompt:
        continue
    if prompt in ("/quit", "/exit", "/q"):
        break
    if prompt.startswith("/temp"):
        temp = float(prompt.split()[1]); print(f"  temperature = {temp}"); continue
    if prompt.startswith("/tokens"):
        ntok = int(prompt.split()[1]); print(f"  tokens = {ntok}"); continue
    print("gpt \u25b8 ", end="", flush=True)
    for delta in tiny_gpt.stream(model, tok, cfg, prompt, n_new=ntok, temperature=temp):
        print(delta, end="", flush=True)
    print()